# Lecture 9: policy experiments, welfare, and the transition

**ECU44291 Quantitative Macroeconomics, Week 10.**

Policy experiments are kind of why we got in the game. The value of quantitative macro is to be able to provide some kind of answer of whether one scenario is 'better' than another.  Today we see two ways to pay for a pension, one way of assessing which situation makes people "better off" (the consumption equivalent), and what happens to the people alive when the reform arrives. Most everything here is the two-period OLG economy from Lectures 2 and 7, where each step is a line or two. The sixty-cohort results are on the slides; building that economy is milestone 3.

This is the complete notebook, and I walk through most of it in lecture.

## The two-period OLG economy, with two kinds of pension

As in Lecture 7: log utility, $\beta = 0.9$, $\alpha = 0.3$, $\delta = 0$, $\tau_L = 0.2$, $\tau_K = 0.15$, one young and one old household, $L = 1$. A period is a generation, about 30 years.

Two pension schemes, each paid for by a contribution on the wage of the young:

- **pay-as-you-go**, rate $\tau_p$: this period's contributions are paid out to this period's old, so the old receive $b = \tau_p w$;
- **funded**, rate $\tau_f$: the contribution $\tau_f w$ goes into an account that is invested in capital, and is paid back with its return, $R\,\tau_f w$, when the household is old.

In [1]:
import numpy as np
from scipy.optimize import brentq

def params(**overrides):
    p = dict(alpha=0.3, beta=0.9, delta=0.0, tau_L=0.2, tau_K=0.15, tau_p=0.0, tau_f=0.0)
    p.update(overrides)
    return p

def prices(K, p):
    """Wage and the household's after-tax gross return at capital K (L = 1)."""
    w = (1 - p["alpha"]) * K ** p["alpha"]
    R = 1 + (1 - p["tau_K"]) * (p["alpha"] * K ** (p["alpha"] - 1) - p["delta"])
    return w, R

def saving(w, R, p):
    """Private saving of the young (log utility). The funded account is saved for them on top of this."""
    income_y = (1 - p["tau_L"] - p["tau_p"] - p["tau_f"]) * w
    income_o = p["tau_p"] * w + R * p["tau_f"] * w        # pension when old: pay-as-you-go, plus the account
    return (p["beta"] * R * income_y - income_o) / ((1 + p["beta"]) * R)

def steady_state(p):
    """Capital is private saving plus the funded account: K = s + tau_f w."""
    def gap(K):
        w, R = prices(K, p)
        return saving(w, R, p) + p["tau_f"] * w - K
    K = brentq(gap, 1e-3, 2.0, xtol=1e-12)
    w, R = prices(K, p)
    s = saving(w, R, p)
    account = p["tau_f"] * w
    c_y = (1 - p["tau_L"] - p["tau_p"] - p["tau_f"]) * w - s
    c_o = R * (s + account) + p["tau_p"] * w
    return dict(K=K, w=w, R=R, s=s, account=account, c_y=c_y, c_o=c_o)

## Task 1: pay-as-you-go against funded, in steady state

Both schemes take 30% of the young's wage.

**Question:** compared with an economy with no pension at all, will the funded economy have more capital, less, or the same?

In [2]:
schemes = {"no pension": params(), "pay-as-you-go": params(tau_p=0.3), "funded": params(tau_f=0.3)}
ss = {name: steady_state(p) for name, p in schemes.items()}

print(f"{'':16s}" + "".join(f"{name:>15s}" for name in ss))
for row in ("K", "w", "R", "s", "account", "c_y", "c_o"):
    print(f"{row:16s}" + "".join(f"{v[row]:15.4f}" for v in ss.values()))

                     no pension  pay-as-you-go         funded
K                        0.1502         0.0535         0.1502
w                        0.3964         0.2907         0.3964
R                        1.9613         2.9811         1.9613
s                        0.1502         0.0535         0.0313
account                  0.0000         0.0000         0.1189
c_y                      0.1669         0.0919         0.1669
c_o                      0.2946         0.2466         0.2946


*Answer:* The same. The household wanted to save $\frac{\beta}{1+\beta}(1-\tau_L)\,w$ in total. The account saves $\tau_f w$ of that for it, at the same return, so it saves exactly that much less on its own: private saving falls one for one. Capital, prices and consumption are all as with no pension. (This needs the account to be no bigger than what the household would save anyway: here 0.3 against 0.379 of the wage.)

Pay-as-you-go is different: the contribution is paid straight to today's old, so it is never invested. Capital falls by almost two thirds.

## Task 2: One number that defines what we mean by "better off"

Lifetime utility of a household born into each steady state is $U = \ln c_y + \beta \ln c_o$. Utility has no units, so we convert the difference into consumption: the **consumption equivalent** $\lambda$ is the percentage by which you would have to raise the pay-as-you-go household's consumption, in both periods, to make it as well off as the funded household. With log utility,
$$\ln\big((1+\lambda) c_y\big) + \beta \ln\big((1+\lambda) c_o\big) = U + (1+\beta)\ln(1+\lambda), \qquad\text{so}\qquad \lambda = \exp\!\Big(\frac{U_{\text{new}} - U_{\text{base}}}{1+\beta}\Big) - 1 .$$

**Question:** is $\lambda$ for funded against pay-as-you-go above or below 10%?

In [3]:
def lifetime_utility(c_y, c_o, p):
    return np.log(c_y) + p["beta"] * np.log(c_o)

def consumption_equivalent(U_new, U_base, p):
    return np.exp((U_new - U_base) / (1 + p["beta"])) - 1

p = params()
U = {name: lifetime_utility(v["c_y"], v["c_o"], p) for name, v in ss.items()}
lam = consumption_equivalent(U["funded"], U["pay-as-you-go"], p)
print(f"lifetime utility: pay-as-you-go {U['pay-as-you-go']:.4f}, funded {U['funded']:.4f}")
print(f"consumption equivalent of funded against pay-as-you-go: {lam:+.4f}")

# check: scale pay-as-you-go consumption by (1 + lam) in both periods
base = ss["pay-as-you-go"]
print(f"check: {lifetime_utility((1 + lam) * base['c_y'], (1 + lam) * base['c_o'], p):.4f}")

lifetime utility: pay-as-you-go -3.6469, funded -2.8903
consumption equivalent of funded against pay-as-you-go: +0.4892
check: -2.8903


A newborn in the funded steady state is as well off as a pay-as-you-go newborn whose consumption was 49% higher in both periods. Large, because a period is 30 years and the pay-as-you-go pension cut capital by almost two thirds (these results are a bit wonky in two-period world).

## Task 2, continued: the same number from $V$, on a sixty-age household

With sixty ages there is no closed form for lifetime utility, but your solver already computes it: the lifetime utility of a newborn with no assets is $V_1(0)$, the first row of `V` at the first grid point, `V[0, 0]`. With $\sigma = 2$, $\lambda = V^{\text{base}}_1(0) / V^{\text{reform}}_1(0) - 1$.

Here: Lecture 3's household on the grid, at fixed prices, $r = 5.45\%$ and $w = 1.058$ (the pay-as-you-go steady state on the slides; finding them is M3). Pay-as-you-go with repl $= 0.4$, so with equal cohorts the budget needs $\tau = 0.4 \times 15/45$. Funded: the same $\tau$ into an account that earns $r$ and is paid back as an annuity $P$ over the 15 years of retirement.

**Before you run:** at these prices, is the funded household better off, and by more or less than the 49% in two periods?

In [4]:
J, J_R, beta, sigma, repl = 60, 45, 0.96, 2.0, 0.4
r, w = 0.0545, 1.058                       # the pay-as-you-go steady state's prices (slides); M3 finds them
R = 1 + r
age = np.arange(1, J + 1)
e = np.exp(0.06 * age - 0.0012 * age ** 2)
e = e / e[:J_R].mean()
e[J_R:] = 0.0
tau = repl * (J - J_R) / J_R               # the pay-as-you-go budget with equal cohorts

def solve_household(y, a_max=25.0, n_a=300):
    """Lecture 3's household on the grid, for an income profile y. Returns V."""
    a = np.linspace(0.0, a_max, n_a)
    V = np.zeros((J + 1, n_a))
    for j in range(J - 1, -1, -1):
        c = R * a[:, None] + y[j] - a[None, :]
        u = np.full(c.shape, -np.inf)
        u[c > 0] = c[c > 0] ** (1 - sigma) / (1 - sigma)
        V[j] = (u + beta * V[j + 1][None, :]).max(axis=1)
    return V

# pay-as-you-go: contribute tau while working, receive repl * w when retired
y_payg = (1 - tau) * w * e
y_payg[J_R:] = repl * w

# funded: the same contribution into an account earning r, paid back as an annuity P
F = 0.0
for j in range(J_R):
    F = R * F + tau * w * e[j]
n = J - J_R
P = R ** n * F / sum(R ** k for k in range(n))     # runs the account down to zero at the end of life
y_funded = (1 - tau) * w * e
y_funded[J_R:] = P

V_payg = solve_household(y_payg)[0, 0]
V_funded = solve_household(y_funded)[0, 0]
print(f"contribution {tau:.4f}; pension each year: pay-as-you-go {repl * w:.3f}, funded {P:.3f}")
print(f"V_1(0): pay-as-you-go {V_payg:.3f}, funded {V_funded:.3f}")
print(f"consumption equivalent of funded against pay-as-you-go: {V_payg / V_funded - 1:+.4f}")

contribution 0.1333; pension each year: pay-as-you-go 0.423, funded 2.378
V_1(0): pay-as-you-go -27.559, funded -25.736
consumption equivalent of funded against pay-as-you-go: +0.0708


About 7%: the same contributions earn $r$ instead of nothing, so the pension each year is 2.38 instead of 0.42, and the household saves less of its own. Smaller than in two periods, because this is only the return, at prices that do not move. The slides show what happens when they do.

Both numbers compare households born long after any reform. Nobody alive when the reform happens is in them.

## Task 3: the reform as a path

The economy has been pay-as-you-go for a long time. At date 0 the government announces, as a surprise, that from now on pensions are funded. By Task 1, a funded pension is the same as saving for yourself, so from date 0 each young household simply saves for its own old age.

The old at date 0 paid into pay-as-you-go when they were young, and expect $b = \tau_p w_0$. Someone has to pay it. Here the young at date 0 pay them one last time (`honour=True`), and also save for themselves.

Capital at date 0 is what the old saved when young, so it is the pay-as-you-go steady state's. From there the path runs forward one date at a time: prices from $K_t$; the young save by the rule; their saving is $K_{t+1}$. With log utility and no pension when old, the rule needs only today's wage, so no future price has to be guessed.

**Question:** is the household young at the reform better or worse off than if pay-as-you-go had stayed?

In [5]:
def transition(p_old, honour=True, dates=8):
    """Pay-as-you-go before date 0 (rate p_old["tau_p"]); each young household saves for itself from date 0."""
    old_ss = steady_state(p_old)
    K, s_prev = old_ss["K"], old_ss["s"]
    path = []
    for t in range(dates):
        w, R = prices(K, p_old)
        last = p_old["tau_p"] * w if (t == 0 and honour) else 0.0      # the last pay-as-you-go pension
        c_o = R * s_prev + last                                         # the old at date t
        s = p_old["beta"] / (1 + p_old["beta"]) * ((1 - p_old["tau_L"]) * w - last)
        c_y = (1 - p_old["tau_L"]) * w - last - s
        path.append(dict(t=t, K=K, w=w, R=R, last=last, s=s, c_y=c_y, c_o=c_o))
        K, s_prev = s, s
    # welfare by birth cohort, against staying in pay-as-you-go: born at t, young at t, old at t + 1
    U_base = lifetime_utility(old_ss["c_y"], old_ss["c_o"], p_old)
    lam = {-1: consumption_equivalent(lifetime_utility(old_ss["c_y"], path[0]["c_o"], p_old), U_base, p_old)}
    for t in range(dates - 1):
        lam[t] = consumption_equivalent(lifetime_utility(path[t]["c_y"], path[t + 1]["c_o"], p_old), U_base, p_old)
    return path, lam

path, lam = transition(params(tau_p=0.3), honour=True)
print("date      K       w       R   last pension     s")
for r in path[:6]:
    print(f"{r['t']:4d}  {r['K']:.4f}  {r['w']:.4f}  {r['R']:.4f}      {r['last']:.4f}    {r['s']:.4f}")
print("\nconsumption equivalent by birth date (-1 is old at the reform, 0 is young at it):")
print("  ".join(f"{t}: {v:+.3f}" for t, v in lam.items()))

date      K       w       R   last pension     s
   0  0.0535  0.2907  2.9811      0.0872    0.0689
   1  0.0689  0.3137  2.6594      0.0000    0.1189
   2  0.1189  0.3695  2.1324      0.0000    0.1400
   3  0.1400  0.3881  2.0097      0.0000    0.1471
   4  0.1471  0.3939  1.9756      0.0000    0.1493
   5  0.1493  0.3956  1.9656      0.0000    0.1499

consumption equivalent by birth date (-1 is old at the reform, 0 is young at it):
-1: +0.000  0: -0.211  1: +0.226  2: +0.404  3: +0.463  4: +0.481  5: +0.487  6: +0.488


The young at the reform pay twice: the old's pension, and their own saving. They are 21% worse off. Every later cohort gains, more and more as capital builds up, towards the steady state's 49%. The old at the reform are untouched.

A steady-state comparison reports only the 49%.

## Task 4: who pays for the last pension

**The edit:** in the cell above, change `honour=True` to `honour=False`. The pension stops at date 0, and the old at the reform get only their own savings.

**Question:** who loses now, and who gains compared with Task 3?

With `honour=False`, the old at the reform lose 19% and the young at the reform gain 15%. The cost of ending pay-as-you-go has not gone away: it has moved from one generation to the other. Spreading it over later cohorts with government debt would move it again; it cannot make it disappear.

## Task 5 (homework): the other direction

Start from the steady state with **no** pension. At date 0, as a surprise, a pay-as-you-go pension with $\tau_p = 0.3$ starts: from date 0 the young pay $\tau_p w_t$ and receive $\tau_p w_{t+1}$ when old, and the old at date 0 receive $\tau_p w_0$ without having paid anything.

Compute the path of $K$ and the consumption equivalent of each birth cohort against staying with no pension.

One difference from Task 3: the young now get a pension when old, $\tau_p w_{t+1}$, and the rule's second term needs $w_{t+1}$ and $R_{t+1}$, which depend on $K_{t+1}$, which is their own saving. So at each date, find the saving $s_t$ that is consistent with the prices it creates, with `brentq`.

In [ ]:
# Solution posted next week. Try it yourself first.


## Before the next lecture

- **M3, the engine and its calibration: Friday 27 November, 17:00**, tag `m3`.
- **Tutorial 4, Wednesday 25 November:** bring one figure from your extension.
- Task 5 above.
- Next lecture, Thursday 26 November: income risk in the life-cycle household.